In [ ]:
#| include: false

%load_ext autoreload
%autoreload 2

# Introduction and Theory

The nucleophilic substitution reaction ($S_\mathrm{N}2$) provides a compact model for studying bond formation, bond breaking, and transition-state geometry. We consider the symmetric chloride-exchange reaction

$$
    \mathrm{Cl_A^- + CH_3Cl_B \rightarrow Cl_A CH_3 + Cl_B^-}   
$$

The labels $\mathrm{Cl_A}$ and $\mathrm{Cl_B}$ distinguish the incoming and leaving chlorine atoms throughout the calculation, even though the two atoms are chemically identical. 

We study the gas-phase reaction first, with a total charge of $-1$ and a singlet electronic state. Solvent effects are outside the scope of this initial calculation.

Our aim is to construct and characterize a minimum energy path (MEP) by using ORCA to evaluate electronic energies and gradients, while implementing the Nudged Elastic Band (NEB) and Climbing Image NEB (CI-NEB) optimization in Python. Specifically, we will:

1. Optimize the reactant and product configurations, then create initial images by linear interpolation.
2. Request electronic energies and Cartesian gradients from ORCA for the intermediate images.
3. Optimize the band using our own NEB implementation, followed by CI-NEB.
4. Refine and verify the proposed transition state by Hessian and vibrational analysis, and compare it with an ORCA transition-state optimization.
5. Examine molecular orbital energies and shapes along the converged reaction pathway.

This chapter introduces the chemical motivation, the mathematical formulation of MEPs, the discretized NEB algorithm, and the electronic structure concepts needed for the subsequent calculations.

## Reaction Background and Motivation

An $S_\mathrm{N}2$ reaction occurs through a backside attack: the incoming nucleophile approaches carbon opposite the leaving group. The incoming $\ce{C-Cl}$ bond forms as the original $\ce{C-Cl}$ bond breaks. Near the ideal symmetric transition-state geometry, the two $\ce{C-Cl}$ distances are approximately equal and the three hydrogen atoms lie close to a plane perpendicular to the $\ce{Cl-C-Cl}$ axis.

For our atom labels, define the two distances and reaction coordinate:

$$
    d_A(\mathbf R)=\|\mathbf r_{\mathrm{Cl_A}}-\mathbf r_{\mathrm C}\|\qquad
    d_B(\mathbf R)=\|\mathbf r_{\mathrm{Cl_B}}-\mathbf r_{\mathrm C}\|
$$

$$
    \boxed{q(\mathbf R)=d_A(\mathbf R)-d_B(\mathbf R)}
$$

The reactant side typically has $q>0$, whereas the product side has $q<0$. The symmetric transition-state region occurs near $q=0$, but **$q=0$ is not by itself a transition-state criterion**: the full molecular geometry and energy gradient must also be considered.

An energy barrier separates the two sides of the reaction path. Locating the transition state provides a geometric and electronic picture of the bond-exchange process. Ordinary energy minimization typically converges to a stable reactant or product complex, rather than to the unstable transition-state geometry. A path-based optimization method offers a way to approach the relevant saddle point.

For a gas-phase ion--molecule reaction, the endpoints used in fixed-endpoint NEB should be defined consistently (for example, optimized entrance and exit complexes at specified finite separations). A separated reactant and product at effectively infinite distance should not be represented by arbitrary finite Cartesian structures without explaining that approximation.

A molecular configuration is represented by the Cartesian coordinates of its \(N\) atoms:

$$
    \mathbf R=(\mathbf r_1,\mathbf r_2,\ldots,\mathbf r_N)\in\mathbb R^{3N}
$$

The potential energy surface (PES) is a scalar function

$$
    E:\mathbb R^{3N}\rightarrow\mathbb R
$$

and the physical force is the negative energy gradient:

$$
    \mathbf F(\mathbf R)=-\nabla E(\mathbf R)
$$

Our objective is to find a minimum energy path (MEP) connecting the reactant and product configurations.


### Continuous Formulation 


We represent the reaction pathway by a continuous, differentiable curve

$$
    \gamma:[0,1]\rightarrow\mathbb R^{3N}
$$

with fixed endpoints

$$
    \gamma(0)=\mathbf R_{\mathrm{reactant}}
    \qquad
    \gamma(1)=\mathbf R_{\mathrm{product}}
$$

The unit tangent vector along the pathway is

$$
    \hat{\boldsymbol\tau}(s)
    =
    \frac{\gamma'(s)}{\|\gamma'(s)\|}
$$

The energy gradient can be decomposed into parallel and perpendicular components:

$$
    \nabla E(\gamma(s))
    =
    \nabla E(\gamma(s))_{\parallel}
    +
    \nabla E(\gamma(s))_{\perp}
$$

where

$$
    \nabla E(\gamma(s))_{\parallel}
    =
    \left(
    \nabla E(\gamma(s))\cdot\hat{\boldsymbol\tau}(s)
    \right)\hat{\boldsymbol\tau}(s)
$$

$$
    \nabla E(\gamma(s))_{\perp}
    =
    \nabla E(\gamma(s))
    -
    \nabla E(\gamma(s))_{\parallel}
$$

For a minimum energy path, the perpendicular component vanishes:

$$
    \nabla E(\gamma(s))_{\perp}=0
$$

This condition applies at regular interior points of the curve, where $\gamma'(s)\ne0$

Equivalently, the energy gradient must be parallel to the pathway:

$$
    \nabla E(\gamma(s))\parallel\hat{\boldsymbol\tau}(s)
$$  

Thus, an MEP is not necessarily a path on which the energy is minimized along the tangent direction. Instead, the energy is stationary in directions perpendicular to the path.


### Discrete Formulation

For computation, we discretize the pathway into $M$ images:

$$
    \{\mathbf R_i\}_{i=0}^{M-1}
$$

where

$$
    \mathbf R_0=\mathbf R_{\mathrm{reactant}}
    \qquad
    \mathbf R_{M-1}=\mathbf R_{\mathrm{product}}
$$

Only the intermediate images $i=1,\ldots,M-2$ are updated.

For each intermediate image, the Nudged Elastic Band (NEB) force is defined as

$$  
    \mathbf F_i^{\mathrm{NEB}}
    =
    -\nabla E(\mathbf R_i)_{\perp}
    +
    \mathbf F_{i,\parallel}^{\mathrm{spring}}
$$

The perpendicular energy gradient is

$$
    \nabla E(\mathbf R_i)_{\perp}
    =
    \nabla E(\mathbf R_i)
    -
    \left(
    \nabla E(\mathbf R_i)\cdot\hat{\boldsymbol\tau}_i
    \right)\hat{\boldsymbol\tau}_i
$$

The spring force acts along the local tangent:

$$
    \mathbf F_{i,\parallel}^{\mathrm{spring}}
    =
    k(d_i-d_{i-1})\hat{\boldsymbol\tau}_i
$$

where

$$
    d_i=\|\mathbf R_{i+1}-\mathbf R_i\|
$$

Here, $k$ is the spring constant and $\hat{\boldsymbol\tau}_i$ is the discrete unit tangent vector.

The perpendicular physical force relaxes the pathway toward the MEP, while the parallel spring force controls the spacing between images.


#### Climbing Image NEB (CI-NEB)


After the conventional NEB pathway is approximately converged, we select the highest-energy intermediate image:

$$
    i_{\mathrm{CI}}
    =
    \underset{1\le i\le M-2}{\operatorname{argmax}}\ E(\mathbf R_i)
$$

For this image, the spring force is removed and the parallel component of the physical force is reversed:

$$
    \mathbf F_{i_{\mathrm{CI}}}^{\mathrm{CI}}
    =
    -\nabla E(\mathbf R_{i_{\mathrm{CI}}})
    +
    2\left(
    \nabla E(\mathbf R_{i_{\mathrm{CI}}})
    \cdot\hat{\boldsymbol\tau}_{i_{\mathrm{CI}}}
    \right)
    \hat{\boldsymbol\tau}_{i_{\mathrm{CI}}}
$$

This drives the climbing image toward a first-order saddle point along the reaction pathway.


### Numerical Implementation


#### Initial Images


The initial pathway is generated using linear interpolation between the reactant and product geometries:

$$
    \mathbf R_i^{(0)}
    =
    (1-t_i)\mathbf R_{\mathrm{reactant}}
    +
    t_i\mathbf R_{\mathrm{product}},
    \qquad
    t_i=\frac{i}{M-1}
$$

This provides the initial images before NEB optimization.



#### Tangent Calculation


For each intermediate image, define the forward and backward displacement vectors:

$$
    \boldsymbol\Delta_i^+
    =
    \mathbf R_{i+1}-\mathbf R_i,
    \qquad
    \boldsymbol\Delta_i^-
    =
    \mathbf R_i-\mathbf R_{i-1}
$$

We use an energy-weighted tangent to improve the stability of the NEB calculation.

If the energy increases monotonically,

$$
    E_{i+1}>E_i>E_{i-1}
$$

we choose

$$
    \boldsymbol\tau_i=\boldsymbol\Delta_i^+
$$

If the energy decreases monotonically,

$$
    E_{i+1}<E_i<E_{i-1} 
$$

we choose

$$
    \boldsymbol\tau_i=\boldsymbol\Delta_i^-
$$

Otherwise, define

$$
    \Delta E_i^+
    =
    |E_{i+1}-E_i|
    \qquad
    \Delta E_i^-
    =
    |E_{i-1}-E_i|
$$

$$
    \Delta E_i^{\max}
    =
    \max(\Delta E_i^+,\Delta E_i^-),
    \qquad
    \Delta E_i^{\min}
    =
    \min(\Delta E_i^+,\Delta E_i^-)
$$

For $E_{i+1}>E_{i-1}$,

$$
    \boldsymbol\tau_i
    =
    \boldsymbol\Delta_i^+\Delta E_i^{\max}
    +
    \boldsymbol\Delta_i^-\Delta E_i^{\min}
$$

For $E_{i+1}<E_{i-1}$,

$$
    \boldsymbol\tau_i
    =
    \boldsymbol\Delta_i^+\Delta E_i^{\min}
    +
    \boldsymbol\Delta_i^-\Delta E_i^{\max}
$$

In the remaining case, we use

$$
    \boldsymbol\tau_i
    =
    \boldsymbol\Delta_i^+
    +
    \boldsymbol\Delta_i^-
$$

Finally, normalize the tangent:

$$
    \hat{\boldsymbol\tau}_i
    =
    \frac{\boldsymbol\tau_i}{\|\boldsymbol\tau_i\|}
$$


#### Position Update

At iteration $n$, we calculate the energy and gradient of each intermediate image using ORCA:

$$
    E_i^{(n)}=E(\mathbf R_i^{(n)})
    \qquad
    \mathbf g_i^{(n)}=\nabla E(\mathbf R_i^{(n)})
$$

The NEB force is then calculated from these quantities.

Using a simple explicit update rule,

$$
    \mathbf R_i^{(n+1)}
    =
    \mathbf R_i^{(n)}
    +
    \alpha\mathbf F_i^{\mathrm{NEB}}
$$      

where $\alpha>0$ is the numerical step parameter. During CI-NEB, the climbing image uses $\mathbf F_i^{\mathrm{CI}}$ instead.

To prevent excessively large updates, we impose a maximum displacement $\delta_{\max}$:

$$
    \Delta\mathbf R_i
    =
    \alpha\mathbf F_i
$$

$$
    \Delta\mathbf R_i
    \leftarrow
    \Delta\mathbf R_i
    \min\left(
    1,\frac{\delta_{\max}}{\|\Delta\mathbf R_i\|}
    \right)
$$

The endpoints remain fixed throughout the optimization:

$$
    \mathbf R_0^{(n+1)}=\mathbf R_0^{(n)},
    \qquad
    \mathbf R_{M-1}^{(n+1)}=\mathbf R_{M-1}^{(n)}.
$$



#### Convergence



We monitor the maximum NEB force magnitude over all intermediate images:

$$
    F_{\max}^{(n)}
    =
    \max_{1\le i\le M-2}
    \|\mathbf F_i^{(n)}\|
$$

The optimization is considered converged when

$$
    F_{\max}^{(n)}<\varepsilon
$$

where $\varepsilon$ is a chosen force tolerance.

Once the CI-NEB calculation converges, the climbing image is used as an initial estimate of the transition-state geometry.

The transition state can then be refined and verified using the Hessian and vibrational frequencies.

# Computational Workflow

## Dependencies and numerical conventions

Let $M$ be the number of images, each with $N=6$ atoms. 

Our image array has shape $(M,N,3)$. 

Cartesian positions are expressed in $\mathrm{\AA}$, energies in $\mathrm{eV}$, and NEB forces in $\mathrm{eV/\AA}$.

ORCA's `.engrad` file provides total energy in Hartree and gradient in Hartree/Bohr. Conversion is essential:

$$ E_{\rm eV}=27.2114 E_{\rm Eh} \qquad
\mathbf g_{\rm eV/\AA}=\frac{27.2114}{0.529177}\,\mathbf g_{\rm Eh/a_0}$$

The numerical step parameter in $\mathbf R^{(n+1)}=\mathbf R^{(n)}+\alpha\mathbf F^{\rm NEB}$ therefore has units $\mathrm{\AA^2/eV}$, and the spring constant $k$ has units $\mathrm{eV/\AA^2}$.

In [ ]:
from pathlib import Path
import os
import shutil
from neb.neb_core import *
from neb.neb_runner import *
from neb.neb_analysis import *
from neb.orbital_analysis import *
from neb.orca import *
from neb.util import *
from neb.visualization import *
from neb.paths import *
from neb.initial_path import *


# Experiment

REACTION = "sn2_cl_ch3cl"
ENVIRONMENT = "gas_phase"

DATA = experiment_path(REACTION, ENVIRONMENT)

assert DATA.is_dir(), f"Experiment not found: {DATA}"

print("Project root:", ROOT)
print("Experiment:", DATA)


# Unit conversion

EH_TO_EV    = 27.211386245988
BOHR_TO_ANG = 0.529177210903
GRAD_FACTOR = EH_TO_EV / BOHR_TO_ANG


# Calculation settings

ATOMS = ("C", "H", "H", "H", "Cl", "Cl")

CHARGE = -1
MULT = 1

METHOD = "B3LYP"
BASIS = "def2-SVP"

NPROCS = 6
N_IMAGES = 9


# ORCA config

ORCA_EXE = os.environ.get("ORCA_EXE",shutil.which("orca") or "",)

ORCA_SETTINGS = dict(
    executable=ORCA_EXE,
    method=METHOD,
    basis=BASIS,
    charge=CHARGE,
    mult=MULT,
    nprocs=NPROCS,
)

print(
    "ORCA_EXE:",
    ORCA_EXE if ORCA_EXE else "Not found"
)

In [ ]:
#| include: false

# Calculation switches

RUN_ENDPOINTS = False
RUN_TEST      = False
RUN_NEB       = False
RUN_CI        = False
RUN_ANALYSIS  = False
DRAW          = False
RUN_TS_FREQ   = False
ORBIT         = False

## 2.2 Construct reactant and product



We use six atoms in a fixed order: $[\mathrm C,\mathrm H_1,\mathrm H_2,\mathrm H_3,\mathrm{Cl_A},\mathrm{Cl_B}]$.

- Reactant: incoming $\mathrm{Cl_A^-}$ on the negative $z$ side, bonded $\mathrm{Cl_B}$ on the positive side.
- Product: $\mathrm{Cl_A}$ bonded on negative $z$ side, leaving $\mathrm{Cl_B^-}$ on positive side.
- The three hydrogen atoms invert their pyramidal orientation in the transition.

In [ ]:
angles = np.deg2rad([0.0, 120.0, 240.0])

def make_guess(z_a, z_b, h_z, ch=1.09):
    r_xy = np.sqrt(ch**2 - h_z**2)
    hydrogens = np.column_stack((r_xy*np.cos(angles),
                                 r_xy*np.sin(angles),
                                 np.full(3, h_z)))
    return np.vstack(([0.,0.,0.], 
                      hydrogens,
                      [0.,0.,z_a], 
                      [0.,0.,z_b]))

reactant_guess = make_guess(-3.50, 1.80, -0.32)
product_guess  = make_guess(-1.80, 3.50,  0.32)

for name, coords in [
    ("reactant_guess", reactant_guess),
    ("product_guess", product_guess),
]:
    file = DATA / "endpoints" / f"{name}.xyz"

    if not file.exists():
        write_xyz(
            file,
            ATOMS,
            coords,
            name
        )

view = show_molecules(ATOMS, 
                      reactant_guess, 
                      product_guess, 
                      width=500, 
                      height=200, 
                      linked=True)

## 2.3 ORCA and input generation

Use one ORCA task per directory, and the code checks normal termination and parses the `.engrad` section markers rather than assuming the gradient appears at a hard-coded line number.

ORCA's `Opt` geometry output is typically `basename.xyz`; an ORCA normal termination message by itself does not prove geometry optimization converged, so the optimization completion marker is also checked.

In [ ]:
print(
    orca_input(
        reactant_guess,
        ATOMS,
        task="Opt",
        method=METHOD,
        basis=BASIS,
        charge=CHARGE,
        mult=MULT,
        nprocs=NPROCS
    )
)

### Optimize two endpoints


The full system has total charge $-1$ and multiplicity $1$ in both endpoints. We optimize the whole ion–molecule complex, rather than separately optimizing isolated fragments. 

This avoids mixing an infinite-separation endpoint with a finite-distance endpoint. Check that the optimization preserves the desired reactant/product basin; if a chlorine migrates, use a better initial geometry or appropriate endpoint constraints.

In [ ]:
output_file = DATA / "endpoints" / "optimized.npz"

if RUN_ENDPOINTS:
    reactant = optimize_endpoint(
        "reactant", reactant_guess,
        ATOMS, DATA, **ORCA_SETTINGS
    )

    product = optimize_endpoint(
        "product", product_guess,
        ATOMS, DATA, **ORCA_SETTINGS
    )

    np.savez(
        output_file,
        reactant=reactant,
        product=product
    )

    print("Both endpoint optimizations converged.")

else:
    with np.load(output_file) as data:
        reactant = data["reactant"]
        product = data["product"]

    print("Loaded previously optimized endpoints.")

## Generate Initial Path

### Linear Interpolation

Once ORCA-optimized endpoints are available, generate $M$ images using the **existing** `linear()` function:

$$
    \mathbf R_i^{(0)}=\left(1-\frac{i}{M-1}\right)\mathbf R_R+\frac{i}{M-1}\mathbf R_P
$$

For a symmetric SN2 reaction a useful coordinate is $q=d_{\rm C-Cl_A}-d_{\rm C-Cl_B}$. 

$q$ changes from positive to negative across the band. Images that have unphysical interatomic distances should be improved before expensive DFT NEB iterations (e.g. via IDPP), rather than blindly run.

In [ ]:
# 1. Generate initial NEB path

initial_path = DATA / "neb" / "initial_images.npy"

if initial_path.exists():
    initial_images = np.load(initial_path)
    print("Loaded existing initial path.")
else:
    initial_images = gen_images(reactant,
                                product,
                                N_IMAGES,
                                method="idpp")
    initial_path.parent.mkdir(parents=True,
                              exist_ok=True)

    np.save(initial_path, initial_images)
    print("Generated new initial path.")

(DATA / "neb").mkdir(parents=True, exist_ok=True)
np.save(DATA / "neb" / "initial_images.npy", initial_images)

print(f"Generated {len(initial_images)} initial images.")
print("Shape:", initial_images.shape)


# 2. Analyze bond distances and reaction coordinate

da = bond_distances(initial_images, 0, 4)  # C-Cl_A
db = bond_distances(initial_images, 0, 5)  # C-Cl_B
q = reaction_coordinate(initial_images)

dmin = min_pair_distances(initial_images)

# 3. Plot path diagnostics

fig, axs = plt.subplots(1, 2, figsize=(11, 4))

axs[0].plot(da, "o-", label=r"C–Cl$_A$")
axs[0].plot(db, "s-", label=r"C–Cl$_B$")
axs[0].set(
    xlabel="Image index",
    ylabel="Distance (Å)",
    title="Linear interpolation"
)
axs[0].legend()
axs[0].grid(alpha=0.2)

axs[1].plot(q, "o-")
axs[1].axhline(0, color="k", lw=0.7)
axs[1].set(
    xlabel="Image index",
    ylabel="q (Å)",
    title="Reaction coordinate"
)
axs[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

# 4. Check minimum atomic separations

print(
    "Minimum all-atom separation by image (Å):",
    np.round(dmin, 3)
)

print("Check images before initiating DFT NEB.")

# 5. Visualize the initial NEB path

views = show_images(
    ATOMS,
    initial_images,
    titles=[
        f"Image {i} | q = {q[i]:+.2f} Å"
        for i in range(N_IMAGES)
    ],
    columns=3,
    panel_width=250,
    panel_height=250,
    linked=True,
    labels=True
)

### IDPP Interpolation

### Implementation

## 2.5 ORCA energies and gradients with exact-geometry caching


For every internal image, an `EnGrad` job returns $(E_i,\nabla E_i)$. We also compute endpoint energies **once** at their fixed geometries. We cache the results keyed by coordinates, the electronic structure method and related settings, so rerunning a previously evaluated image need not rerun ORCA.

**Unit test:** the included `.engrad` mock tests the parser, but does **not** substitute for an ORCA calculation. Make sure the parser works against one real `.engrad` generated by your installed ORCA version before a long NEB run.

In [ ]:
test_dir = DATA / "parser_test"
test_dir.mkdir(parents=True, exist_ok=True)

mock = test_dir / "mock.engrad"

mock.write_text(
    "# Number of atoms\n"
    "6\n"
    "# The current total energy in Eh\n"
    "-10.0\n"
    "# The current gradient in Eh/bohr\n"
    + "\n".join(["0.01"] * 18)
    + "\n"
)

test_e, test_g = parse_engrad(mock, n_atoms=6)

assert np.isclose(test_e, -10 * 27.211386245988)
assert test_g.shape == (6, 3)

print("Parser mock test passed.")

In [ ]:
from functools import partial

RUN_TEST = False

calculator = partial(
    energy_gradient,
    atoms=ATOMS,
    data_dir=DATA,
    **ORCA_SETTINGS
)

if RUN_TEST:
    energy, gradient = calculator(initial_images[0])

    print("Energy:", energy, "eV")
    print("Gradient shape:", gradient.shape)

## 2.6 Conventional NEB, `update()` function


We use your existing NEB force:

$$ \mathbf F_i^{\rm NEB}= -\nabla E(\mathbf R_i)_\perp+k(d_i-d_{i-1})\hat{\boldsymbol\tau}_i. $$

Your updater applies $\Delta\mathbf R_i=\alpha\mathbf F_i$ with a cap $\|\Delta\mathbf R_i\|\le\delta_{\max}$. To avoid recomputing finished work, checkpoint **every iteration**.

The convergence criterion in your implementation is $F_{\max}=\max_i\|\mathbf F_i\|_2$ across internal images. The force is computed in eV/Å. The default parameters below are **starting values, not guaranteed stable for every potential**; inspect convergence and reduce the step if the band oscillates.

In [ ]:
K_SPRING = 2.0         # eV / Å^2
STEP = 0.005           # Å^2 / eV
MAX_MOVE = 0.025       # Å per full-image step

F_TOL_NEB = 0.15       # eV / Å (initial, moderate tolerance)
F_TOL_CI = 0.08        # eV / Å (stricter)

MAX_ITER_NEB = 250
MAX_ITER_CI = 160

In [ ]:
RUN_NEB = True

if RUN_NEB:
    neb_images, neb_history, neb_ok = run_band(
        stage="neb",
        starting_images=initial_images,
        calculator=calculator,
        data_dir=DATA,
        max_iter=MAX_ITER_NEB,
        tol=F_TOL_NEB,
        climb=False,
        step=STEP,
        k=K_SPRING,
        max_move=MAX_MOVE,
        resume=True
    )

    print("Converged?", neb_ok)

else:
    print("Set RUN_NEB=True to start ORCA + NEB.")


## 2.7 Climbing Image NEB


After conventional NEB, the highest-energy **interior** image is promoted to the climbing image:

$$i_{\rm CI}=\underset{1\le i\le M-2}{\operatorname{argmax}}E(\mathbf R_i),$$

$$\mathbf F_{i_{\rm CI}}^{\rm CI}=-\nabla E(\mathbf R_{i_{\rm CI}})+2\left(\nabla E(\mathbf R_{i_{\rm CI}})\cdot\hat{\boldsymbol\tau}_{i_{\rm CI}}\right)\hat{\boldsymbol\tau}_{i_{\rm CI}}.$$

Only one image is climbed; all others retain conventional NEB forces. As the band evolves, the highest-energy index may change. In this implementation it is recomputed from the current energy array. CI gives a **transition-state candidate** rather than proof of a first-order saddle point.

In [ ]:
RUN_CI = True

if RUN_CI:
    neb_checkpoint = DATA / "neb" / "checkpoint.npz"

    if not neb_checkpoint.exists():
        raise RuntimeError(
            "No NEB checkpoint found. Run conventional NEB first."
        )

    with np.load(neb_checkpoint) as saved:
        neb_images = saved["images"].copy()
        neb_history = saved["history"].copy()

    if len(neb_history) == 0:
        raise RuntimeError("NEB has no iteration history.")

    if neb_history[-1, 1] >= F_TOL_NEB:
        raise RuntimeError(
            "Conventional NEB is not yet converged. "
            f"Current Fmax = {neb_history[-1, 1]:.5f} eV/Å; "
            f"required < {F_TOL_NEB:.5f} eV/Å."
        )

    ci_images, ci_history, ci_ok = run_band(
        stage="ci",
        starting_images=neb_images,
        calculator=calculator,
        data_dir=DATA,
        max_iter=MAX_ITER_CI,
        tol=F_TOL_CI,
        climb=True,
        step=STEP,
        k=K_SPRING,
        max_move=MAX_MOVE,
        resume=True,
        live_plot=True,
        plot_every=5,
        report_every=5
    )

    print("CI converged?", ci_ok)

else:
    print("Set RUN_CI=True after NEB convergence.")

## 2.8 Final energy profile and structural analysis



Use the checkpointed band and compute energy at **every image**. In the symmetric gas-phase model, reactant and product basins should have similar energies (numerical and orientation differences can remain). For a reaction coordinate, show both:

* accumulated configuration-space arc length $s_i=\sum_{j<i}\|\mathbf R_{j+1}-\mathbf R_j\|$;
* chemically interpretable $q_i=d_{\mathrm C-\mathrm{Cl_A}}-d_{\mathrm C-\mathrm{Cl_B}}$.

The barrier reported here is an **electronic energy difference**, not a Gibbs free-energy barrier.

In [ ]:
RUN_ANALYSIS = True

if RUN_ANALYSIS:
    final_images, final_energies, summary, ts_idx, hist_ci = final_analysis(
        data_dir=DATA,
        atoms=ATOMS,
        calculator=calculator,
        force_tol=F_TOL_CI
    )

    display(summary)

    barrier = final_energies[ts_idx] - final_energies[0]

    print(f"TS candidate: Image {ts_idx}")
    print(f"Electronic barrier: {barrier:.5f} eV")

else:
    print("Set RUN_ANALYSIS=True after CI-NEB converges.")

In [ ]:
DRAW = False

if DRAW:
    animation, gif_path = animate_convergence(
        DATA,
        stages=("neb", "ci"),
        tolerance={"neb": F_TOL_NEB, "ci": F_TOL_CI},
        stride=2,
        fps=12,
        save_path=DATA / "convergence.gif",
        show=True
    )

In [ ]:
fig, ax = plot_final_energy_profile(
    DATA,
    stage="ci",
    save_path=DATA / "ci" / "energy_profile.png"
)

In [ ]:
view = show_final_images(
    DATA,
    ATOMS,
    stage="ci",
    columns=3,
    panel_width=250,
    panel_height=250,
    labels=True,
    linked=True
)

In [ ]:
RUN_TS_FREQ = False

if RUN_TS_FREQ:
    # Check final CI results
    ci_file = DATA / "ci" / "final_band.npz"
    ts_file = DATA / "ts" / "ts_candidate.xyz"

    if not ci_file.exists():
        raise RuntimeError("First create the final CI band.")

    if not ts_file.exists():
        raise FileNotFoundError(f"Missing TS candidate: {ts_file}")

    # Read TS candidate coordinates
    ts_coords = read_xyz(ts_file, ATOMS)

    # Generate ORCA frequency input
    freq_input = orca_input(
        ts_coords,
        ATOMS,
        task="Freq",
        method=METHOD,
        basis=BASIS,
        charge=CHARGE,
        mult=MULT,
        nprocs=NPROCS
    )

    # Run ORCA
    job_dir = DATA / "ts" / "freq"

    out = run_orca(
        job_dir,
        freq_input,
        executable=ORCA_EXE
    )

    print("Frequency calculation complete.")
    print("Output:", out)
    print("Hessian:", job_dir / "calc.hess")

else:
    print("Set RUN_TS_FREQ=True after CI convergence.")

In [ ]:
with np.load(DATA / "ci" / "final_band.npz") as saved:
    final_images = saved["images"]

ORBIT = False

if ORBIT:
    orbital_energies = compute_orbitals(
        final_images,
        ATOMS,
        DATA,
        executable=ORCA_EXE,
        method=METHOD,
        basis=BASIS,
        charge=CHARGE,
        mult=MULT,
        nprocs=1,
        grid=50
    )

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

slider = widgets.IntSlider(
    value=4,
    min=0,
    max=len(final_images)-1,
    description="Image"
)

button = widgets.Button(description="Show orbitals")
output = widgets.Output()

def render_orbitals(_):
    with output:
        clear_output(wait=True)
        show_orbitals(
            final_images,
            ATOMS,
            DATA,
            image_index=slider.value,
            isoval=0.035
        )

button.on_click(render_orbitals)

display(widgets.HBox([slider, button]), output)

In [ ]:
view = show_all_orbitals(
    final_images,
    ATOMS,
    DATA,
    images_per_row=2,
    panel_width=240,
    panel_height=260,
    isoval=0.035,
    linked=True,
    labels=False
)